# KITE

The KITE is a loop of two junction-inductor arms threaded by half a flux quantum, shunted by a capacitor [Smith et al., npj Quantum Inf. **6**, 8 (2020)](https://doi.org/10.1038/s41534-019-0231-2).
At $\varphi_\mathrm{ext} = \pi$ the single-Cooper-pair tunneling of the two arms cancels, leaving $\cos 2\varphi$.
This example takes symmetric arms.

In [ ]:
import sympy as sym
import circuit_quantization as cq
from circuit_quantization import Branch, Circuit, Loops, quantize

## Circuit

Node 1 is the island, nodes 2 and 3 are the arm midpoints, and node 4 is ground.

In [ ]:
C_S, C_J, L, E_J = sym.symbols("C_S C_J L E_J", positive=True)
varphi_ext = sym.Symbol("varphi_ext", real=True)

kite = Circuit([
    Branch("CS", 1, 4, C=C_S),
    Branch("J1", 1, 2, C=C_J, EJ=E_J),
    Branch("J2", 1, 3, C=C_J, EJ=E_J),
    Branch("L1", 2, 4, L=L),
    Branch("L2", 3, 4, L=L),
], ground=4)
cq.draw_circuit_diagram(kite, layout={1: (0, 1), 2: (-1, 0), 3: (1, 0)}, grounds="below")

## Loops

The kite loop through both arms carries the external flux; the loop closed by the shunt capacitor is capacitively broken.

In [ ]:
loops = Loops.from_branch_dict(
    kite,
    {"kite": ("J1", "J2", "L1", "L2"), "cb": ("CS", "J1", "L1")},
    {"kite": cq.phase_to_weber(varphi_ext), "cb": 0},
)

## Quantization

In [ ]:
result = quantize(kite, loops, simplify=True)
assert result.coords.types == ("c", "e", "h")
result

The compact island phase (`c`) and the extended arm-difference mode (`e`) share both junction cosines, which for symmetric arms combine into $-2E_J \cos\varphi_1 \cos\varphi_2$.
The harmonic mode (`h`) couples to them only quadratically.
`compact_charge_decoupling` removes the charge coupling between the compact mode and the others, keeping the compact period:

In [ ]:
S = cq.compact_charge_decoupling(result.coords)
decoupled = result.transform(S, simplify=True)
decoupled.ham.E_C